# Tema 12: Programación avanzada - Testing & Debugging
---

## 1. Python es un lenguaje interpretado

Python **no compila** el código antes de ejecutarlo. Esto significa que:

- Los errores se detectan **en tiempo de ejecución**, no antes
- El código que no se ejecuta **no se valida**

Esto tiene consecuencias importantes que debemos entender.

### 1.1 Falta de tipificación

El mismo código puede funcionar o fallar dependiendo de los datos de entrada:

In [ ]:
import random

# Este código a veces funciona y a veces falla
if random.random() > 0.5:
    numero = 2       # int
else:
    numero = "2"     # str

resultado = numero / 2  # Si numero es "2", esto falla
print(resultado)

### 1.2 No se valida el código no ejecutado

Si una función tiene errores pero nunca se llama, Python no se queja:

In [ ]:
def funcion_buena(a):
    b = a.upper()
    return b


def funcion_con_errores():
    # Esta función tiene 3 errores, pero Python no dice nada
    s = un_string + 1         # NameError: 'un_string' no existe
    s.upper(1)                # TypeError: upper() no recibe argumentos
    return valor              # NameError: 'valor' no existe

In [ ]:
# Si solo llamamos a la función buena, todo parece funcionar
entrada = 1

if entrada == 1:
    print(funcion_buena("hola"))  # Funciona
else:
    print(funcion_con_errores())   # Nunca se ejecuta, los errores no se detectan

In [ ]:
# Pero si cambia la entrada...
entrada = 2

if entrada == 1:
    print(funcion_buena("hola"))
else:
    print(funcion_con_errores())   # ¡Ahora sí explota!

**Conclusión**: No podemos fiarnos de que el código "funcione" solo porque no da error. Hay que **probar todas las funciones**.

---

## 2. Testing y control de errores

Cuando nuestro programa depende de `input()`, perdemos mucho tiempo probando manualmente. Es mejor:

1. **Primero**: Probar que la lógica funciona con datos fijos
2. **Después**: Conectar los `input()`

In [ ]:
# ❌ MAL: Probar con input() desde el principio
# nombre = input("Introduce el nombre: ")
# edad_str = input("Edad: ")
# edad = int(edad_str)

# ✅ BIEN: Primero probamos con datos fijos
nombre = "Luis"
edad = 22

# Ahora probamos nuestra lógica
print(f"{nombre} tiene {edad} años")

### 2.1 Pruebas unitarias (testing)

La mejor forma de asegurar que nuestro código funciona es crear un fichero de **tests** que llame a todas las funciones.

```
📁 mi_proyecto/
   ├── funciones.py    <- Nuestras funciones
   ├── main.py         <- Programa principal
   └── tests.py        <- Pruebas de todas las funciones
```

In [8]:
# Ejemplo: funciones.py
def sumar(a: int, b: int) -> int:
    return a + b

def es_mayor_edad(edad: int) -> bool:
    return edad >= 18

def formatear_nombre(nombre: str) -> str:
    return nombre.strip().title()

In [13]:
# Ejemplo: tests.py
# Antes de entregar el código, ejecutamos esto para asegurarnos de que todo funciona

print("Probando sumar()...")
assert sumar(2, 3) == 5, "sumar(2, 3) deberia dar 5"
assert sumar(-1, 1) == 0
assert sumar(0, 0) == 0
print("  ✓ OK")

print("Probando es_mayor_edad()...")
assert es_mayor_edad(18) == True
assert es_mayor_edad(17) == False
assert es_mayor_edad(65) == True
print("  ✓ OK")

print("Probando formatear_nombre()...")
assert formatear_nombre("  luis  ") == "Luis"
assert formatear_nombre("ANA") == "Ana"
assert formatear_nombre("maría garcía") == "María García"
print("  ✓ OK")

print("\n✅ Todos los tests pasaron!")

Probando sumar()...
  ✓ OK
Probando es_mayor_edad()...
  ✓ OK
Probando formatear_nombre()...
  ✓ OK

✅ Todos los tests pasaron!


### 2.2 La función `assert`

`assert` comprueba que una condición sea `True`. Si es `False`, lanza un error.

In [ ]:
# assert que pasa (no hace nada)
assert 2 + 2 == 4
print("Esto se ejecuta porque el assert pasó")

In [ ]:
# assert que falla (lanza AssertionError)
# assert 2 + 2 == 5  # Descomentar para ver el error

---

## 3. Debugging (depuración)

El debugging es seguir el flujo del programa **línea a línea**, viendo el valor de las variables.

### 3.1 Método tradicional: print()

Cuando no sabemos qué falla, ponemos `print()` por todas partes:

In [ ]:
def procesar_datos(datos):
    print(f"DEBUG 1 >>> datos = {datos}")
    print(f"DEBUG 2 >>> type(datos) = {type(datos)}")
    
    resultado = []
    for i, dato in enumerate(datos):
        print(f"DEBUG 3 >>> i={i}, dato={dato}")
        resultado.append(dato * 2)
    
    print(f"DEBUG 4 >>> resultado = {resultado}")
    return resultado

procesar_datos([1, 2, 3])

### 3.2 Método moderno: el depurador (debugger)

En lugar de llenar el código de `print()`, usamos el **depurador** de VS Code:

1. Ponemos un **breakpoint** (punto rojo) en la línea donde queremos parar
2. Ejecutamos en modo **Debug** (F5)
3. El código se detiene y podemos:
   - Ver el valor de todas las variables
   - Avanzar línea a línea (F10)
   - Entrar dentro de funciones (F11)
   - Continuar hasta el siguiente breakpoint (F5)

**💡 Consejo**: No abuses del depurador. Primero piensa analíticamente dónde puede estar el error.

---

## 4. Ejecución de programas Python

### 4.1 De desarrollo a producción

Hasta ahora ejecutamos desde VS Code, pero el objetivo final es ejecutar **sin IDE**, desde la terminal:

```bash
# Navegar al directorio
cd C:\Users\usuario\mi_proyecto

# Ejecutar el programa
python main.py
```

### 4.2 Envío de parámetros desde la terminal: `sys.argv`

En producción, es mejor pasar datos como **argumentos** en lugar de usar `input()`:

```bash
# En lugar de:
python calculadora.py
Introduce número 1: 4
Introduce número 2: 5
Introduce operación: +
Resultado: 9

# Mejor así:
python calculadora.py 4 + 5
9
```

Para leer estos argumentos, usamos `sys.argv`:

In [ ]:
import sys

# sys.argv es una lista con los argumentos de la línea de comandos
# El primer elemento SIEMPRE es el nombre del programa

# Simulamos: python mi_programa.py hola mundo 123
sys.argv = ["mi_programa.py", "hola", "mundo", "123"]

print(f"Nombre del programa: {sys.argv[0]}")
print(f"Argumentos: {sys.argv[1:]}")
print(f"Número de argumentos: {len(sys.argv) - 1}")

In [ ]:
# Ejemplo: calculadora por línea de comandos
import sys

# Simulamos: python calculadora.py 4 + 5
sys.argv = ["calculadora.py", "4", "+", "5"]

if len(sys.argv) == 4:
    numero1 = int(sys.argv[1])
    operador = sys.argv[2]
    numero2 = int(sys.argv[3])
    
    if operador == "+":
        print(numero1 + numero2)
    elif operador == "-":
        print(numero1 - numero2)
    elif operador == "x":  # Usamos 'x' porque '*' tiene significado especial en terminal
        print(numero1 * numero2)
    elif operador == "/":
        print(numero1 / numero2)
else:
    print("Uso: python calculadora.py <num1> <operador> <num2>")

---

## 5. Tipificación de variables

Python permite **indicar tipos** en las variables y funciones. Esto NO obliga a nada, pero ayuda a documentar el código.

### 5.1 Tipificación de variables (no se suele usar)

In [ ]:
# Se puede hacer, pero no aporta mucho porque el tipo se infiere del valor
nombre: str = "Luis"
numero: int = 22

# Python NO valida esto - acepta cualquier valor
nombre: str = 12  # No da error, aunque dijimos que era str
print(nombre, type(nombre))

### 5.2 Tipificación de funciones (muy útil)

En funciones SÍ aporta valor porque define el **interfaz**: qué recibe y qué devuelve.

In [ ]:
def formatear_persona(nombre: str, edad: int) -> str:
    """Formatea los datos de una persona.
    
    Args:
        nombre: Nombre de la persona
        edad: Edad en años
        
    Returns:
        String formateado
    """
    return f"La edad de {nombre} es de {edad} años"


print(formatear_persona("Luis", 22))

### 5.3 Reforzar la tipificación con comprobaciones

Si queremos **forzar** que los tipos sean correctos, podemos comprobarlo manualmente:

In [ ]:
def formatear_persona_segura(nombre: str, edad: int) -> str:
    # Comprobamos los tipos
    if type(nombre) != str or type(edad) != int:
        raise TypeError("Parámetros incorrectos. Se esperaba: (str, int)")
    
    return f"La edad de {nombre} es de {edad} años"


# Prueba con tipos correctos
try:
    print(formatear_persona_segura("Luis", 22))
    print(formatear_persona_segura(22, "Luis"))  # Tipos intercambiados
except TypeError as error:
    print(f"Error: {error}")

### 5.4 Tipos más comunes

```python
def mi_funcion(
    nombre: str,           # String
    edad: int,             # Entero
    precio: float,         # Decimal
    activo: bool,          # Booleano
    notas: list,           # Lista
    datos: dict,           # Diccionario
    coordenadas: tuple     # Tupla
) -> str:                  # Tipo de retorno
    pass
```

---

## 6. Referencias de objetos en memoria

Este es un concepto **muy importante** para entender cómo funciona Python.

### 6.1 Tipos básicos: se copian por valor

In [15]:
# Con int, float, str, bool: se copia el VALOR
numero1 = 1
numero2 = numero1  # Copia el valor

numero1 += 1  # Modificamos numero1

print(f"numero1: {numero1}")  # 2
print(f"numero2: {numero2}")  # 1 (no cambió)

numero1: 2
numero2: 1


### 6.2 Tipos complejos: se copian por referencia

Con listas, diccionarios, etc., lo que se copia es la **dirección de memoria**, no el contenido.

In [14]:
# Con list, dict, etc.: se copia la REFERENCIA
lista1 = [1, 2, 3]
lista2 = lista1  # copia la dirección

lista1.append(4)  # Modificamos lista1

print(f"lista1: {lista1}")  # [1, 2, 3, 4]
print(f"lista2: {lista2}")  # [1, 2, 3, 4] ← ¡También cambió!

lista1: [1, 2, 3, 4]
lista2: [1, 2, 3, 4]


In [16]:
# Podemos verificar que apuntan a la misma dirección de memoria
lista1 = [1, 2, 3]
lista2 = lista1

print(f"Dirección lista1: {hex(id(lista1))}")
print(f"Dirección lista2: {hex(id(lista2))}")
print(f"¿Son el mismo objeto? {lista1 is lista2}")

Dirección lista1: 0x15b86146980
Dirección lista2: 0x15b86146980
¿Son el mismo objeto? True


### 6.3 Cómo hacer una copia real

In [17]:
# Para copiar de verdad, usamos .copy()
lista1 = [1, 2, 3]
lista2 = lista1.copy()  # Crea una nueva lista con los mismos valores

lista1.append(4)

print(f"lista1: {lista1}")  # [1, 2, 3, 4]
print(f"lista2: {lista2}")  # [1, 2, 3] ← No cambió
print(f"¿Son el mismo objeto? {lista1 is lista2}")  # False

lista1: [1, 2, 3, 4]
lista2: [1, 2, 3]
¿Son el mismo objeto? False


### 6.4 En funciones

Este comportamiento es especialmente importante cuando pasamos listas a funciones:

In [ ]:
# Con tipos básicos: la función recibe una COPIA
def duplicar(numero):
    numero = numero * 2
    print(f"Dentro de la función: {numero}")

mi_numero = 5
duplicar(mi_numero)
print(f"Fuera de la función: {mi_numero}")  # No cambió

In [21]:
# Con listas: la función recibe la REFERENCIA (el mismo objeto)
def agregar_elemento(lista):
    lista.append(99)
    print(f"Dentro de la función: {lista}")

mi_lista = [1, 2, 3]
resultado = agregar_elemento(mi_lista)
print(f"Fuera de la función: {mi_lista}")  # ¡SÍ cambió!
print(resultado)

Dentro de la función: [1, 2, 3, 99]
Fuera de la función: [1, 2, 3, 99]
None


In [20]:
mi_lista2 = [1,2,4,3]
mi_lista2.sort()
print(mi_lista2)

[1, 2, 3, 4]


In [ ]:
# Por eso, a veces no es necesario devolver la lista modificada
def agregar_elemento_v2(lista):
    lista.append(99)
    # No hace falta return porque modificamos el objeto original

mi_lista = [1, 2, 3]
agregar_elemento_v2(mi_lista)
print(mi_lista)  # [1, 2, 3, 99]

### 6.5 Resumen: valor vs referencia

| Tipo | Se pasa por | `=` crea | Para copiar |
|------|-------------|----------|-------------|
| `int`, `float`, `str`, `bool` | **Valor** | Copia | No hace falta |
| `list`, `dict`, `set` | **Referencia** | Alias | `.copy()` |

---

## 7. Recordatorio: métodos de string útiles

Algunos métodos muy usados para procesar datos:

In [22]:
# replace() - Reemplazar subcadenas
frase = "La asignatura en plan de Programación en plan me gusta"
limpia = frase.replace("en plan ", "")
print(limpia)

La asignatura de Programación me gusta


In [24]:
# split() - Dividir string en lista
persona = "Luis, 22, Madrid"
partes = persona.split(" ")
print(partes)

nombre, edad, ciudad = partes  # Desempaquetar
print(f"{nombre} tiene {edad} años y vive en {ciudad}")

['Luis,', '22,', 'Madrid']
Luis, tiene 22, años y vive en Madrid


In [ ]:
# Limpiar datos sucios
notas = ["10", "3,2", "4.9", "5 4"]

for i in range(len(notas)):
    # Reemplazamos espacio y coma por punto
    notas[i] = float(notas[i].replace(" ", ".").replace(",", "."))

print(notas)

---

## 📝 Resumen

| Concepto | Descripción |
|----------|-------------|
| **Testing** | Probar todas las funciones con `assert` antes de entregar |
| **Debugging** | Usar el depurador de VS Code en lugar de `print()` |
| **sys.argv** | Lista con los argumentos de línea de comandos |
| **Tipificación** | `def funcion(param: tipo) -> tipo_retorno:` |
| **Valor vs Referencia** | Tipos básicos se copian, listas/dicts se referencian |